# Scheduler Patterns for Finite-Chain TEBD

This notebook is a guide to the scheduler side of the finite-chain TEBD API.

The main idea is simple:

- a **schedule** says where local gates act and in what order
- the **gates** say what gets applied at each scheduled step
- `LocalGateEvolution(...)` packages those two pieces into something `evolve!` can run

We will walk through three common patterns:

1. a standard left-to-right and right-to-left sweep
2. 2-site and 3-site brick schedules
3. a shallow random circuit with mixed gate spans


In [ ]:
using Random
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
using MPSToolkit

default(size=(960, 680), linewidth=2, markersize=4, legend=:best)

spins = spinhalf_matrices()

function local_sz_profile(psi::MPS)
    return real.(expect(psi, "Sz"))
end

function xxz_gate(dt; Jxy::Real=1.0, Delta::Real=1.0)
    h = Jxy * kron(spins.Sx, spins.Sx) +
        Jxy * kron(spins.Sy, spins.Sy) +
        Delta * kron(spins.Sz, spins.Sz)
    return exp(-im * dt * h)
end

function xzx_gate(dt; Jx::Real=0.8, Jz::Real=0.5)
    h = Jx * kron(kron(spins.Sx, spins.Sz), spins.Sx) +
        Jz * kron(kron(spins.Sz, spins.Sz), spins.Sz)
    return exp(-im * dt * h)
end

function random_local_gate(rng::AbstractRNG, span::Int; strength::Real=0.08)
    dim = 2^span
    raw = randn(rng, ComplexF64, dim, dim)
    h = 0.5 * (raw + raw')
    return exp(-im * strength * h)
end


## 1. Standard sweeping gate application

The most direct schedule is a sweep across the chain followed by a sweep back.
For a nearest-neighbor gate on an `nsites`-site chain, the legal starting bonds are `1:(nsites - 1)`.

Here we build the schedule explicitly and apply one sweep per time step.


In [ ]:
nsites = 8
sites = siteinds("S=1/2", nsites)
psi = MPS(sites, n -> isodd(n) ? "Up" : "Dn")

dt = 0.08
gate = xxz_gate(dt)
forward_sweep = collect(1:(nsites - 1))
reverse_sweep = collect((nsites - 1):-1:1)
sweep_schedule = vcat(forward_sweep, reverse_sweep)
sweep_gates = fill(gate, length(sweep_schedule))

profiles = [local_sz_profile(psi)]
for _ in 1:8
    evo = LocalGateEvolution(sweep_gates, dt; schedule=sweep_schedule, nstep=1, maxdim=32, cutoff=1e-10)
    evolve!(psi, evo)
    push!(profiles, local_sz_profile(psi))
end

heatmap(
    0:8,
    1:nsites,
    reduce(hcat, profiles);
    xlabel="sweep step",
    ylabel="site",
    title="Nearest-neighbor sweep schedule",
    colorbar_title="⟨Sz⟩",
)


A few things to notice:

- the schedule is just a vector of bond starts
- `LocalGateEvolution` does not care whether a sweep is left-to-right, right-to-left, or something more exotic
- repeating a pattern is usually easier to read as an outer loop over `evolve!` than as one very long schedule


## 2. Brick-like schedules with 2-site and 3-site gates

Brick schedules are built by interleaving non-overlapping windows.
The same `LocalGateEvolution` type handles both 2-site and 3-site gates, because `tebd_evolve!` infers the gate span from the matrix size.


In [ ]:
psi2 = MPS(sites, n -> n <= nsites ÷ 2 ? "Up" : "Dn")
psi3 = copy(psi2)

gate2 = xxz_gate(0.06)
gate3 = xzx_gate(0.04)

brick2_schedule = [1, 3, 5, 7, 2, 4, 6]
brick3_schedule = [1, 4, 2, 5, 3]

bond_dims_2 = Int[]
bond_dims_3 = Int[]
for _ in 1:6
    evolve!(psi2, LocalGateEvolution(fill(gate2, length(brick2_schedule)), 0.06; schedule=brick2_schedule, maxdim=48, cutoff=1e-10))
    evolve!(psi3, LocalGateEvolution(fill(gate3, length(brick3_schedule)), 0.04; schedule=brick3_schedule, maxdim=48, cutoff=1e-10))
    push!(bond_dims_2, maxlinkdim(psi2))
    push!(bond_dims_3, maxlinkdim(psi3))
end

plot(
    1:6,
    bond_dims_2;
    label="2-site brick",
    xlabel="brick step",
    ylabel="max bond dimension",
    title="Scheduler patterns with different gate spans",
)
plot!(1:6, bond_dims_3; label="3-site brick")


The important part is the schedule itself:

- for 2-site brickwork we alternate odd and even bonds
- for 3-site brickwork we shift the window starts so nearby gates do not all overlap at once

Once the schedule and gate list line up, the driver is unchanged.


## 3. A shallow random circuit with mixed gate spans

A mixed local circuit is a good stress test for the scheduler abstraction because different updates may have different support sizes.
Here we keep the gates shallow and close to the identity so the circuit stays easy for finite-chain TEBD.


In [ ]:
rng = MersenneTwister(7)
psi_random = MPS(sites, n -> isodd(n) ? "Up" : "Dn")

circuit_bonds = Int[]
circuit_gates = Matrix{ComplexF64}[]
circuit_spans = Int[]
for _ in 1:12
    span = rand(rng, 1:3)
    bond = rand(rng, 1:(nsites - span + 1))
    push!(circuit_spans, span)
    push!(circuit_bonds, bond)
    push!(circuit_gates, random_local_gate(rng, span; strength=0.06))
end

evo_random = LocalGateEvolution(circuit_gates, 1.0; schedule=circuit_bonds, maxdim=64, cutoff=1e-10)
evolve!(psi_random, evo_random)

println("scheduled bonds = ", circuit_bonds)
println("gate spans      = ", circuit_spans)
bar(1:nsites, local_sz_profile(psi_random); xlabel="site", ylabel="⟨Sz⟩", title="After one shallow mixed-span circuit", label="local magnetization")


## Closing Notes

All three workflows use the same pattern:

1. prepare a gate list or gate provider
2. prepare a schedule of local starting positions
3. pass both into `LocalGateEvolution`
4. call `evolve!`

That is the key idea to keep in mind when you want to build custom TEBD schedules.
